# Missing Data

## Introduction

Real-world datasets rarely arrive clean. Missing values can prevent type conversions, break summary statistics, and crash ML algorithms. This notebook covers the full toolkit: detecting `NaN`s, spotting hidden placeholder values, finding duplicates, and choosing the right strategy for each case.

## Objectives

You will be able to:

- Detect `NaN` values with `.isna()` and summarise them by column
- Apply the three core strategies — drop, impute, keep — and know when each is appropriate
- Identify missing data disguised as placeholder values (e.g. `?`, `-99`)
- Find exact and key-based duplicate rows
- Choose a handling strategy using the continuous/categorical decision table

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

---

## Part A — Detecting and Handling NaN Values

Pandas represents missing values as `NaN` (Not a Number). They silently propagate through calculations and must be resolved before modelling.

### Detecting NaN

```python
df.isna()          # boolean matrix — True where value is NaN
df.isna().any()    # True/False per column — does this column have any NaN?
df.isna().sum()    # count of NaN per column
```

In [ ]:
df = pd.read_csv('data/dealing_missing_data_lab/titanic.csv')
print(df.shape)
df.isna().sum()

Three columns have missing values. The strategy for each depends on **how much** is missing and **what type** of data it is:

| Column | Missing | Action |
|--------|---------|--------|
| `Cabin` | 687 / 891 = **77%** | Drop the column |
| `Age` | 177 / 891 = **20%** | Impute with median |
| `Embarked` | 2 / 891 = **<1%** | Drop rows |

### Strategy 1 — Drop the column

When a column is missing the majority of its values it contributes little signal. Drop it entirely.

In [ ]:
df.drop(columns='Cabin', inplace=True)
df.isna().sum()

### Strategy 2 — Impute with the median

For continuous columns, replace `NaN` with the **median** rather than the mean. The median is robust to skew and outliers.

> Imputing reduces variance for that feature — keep this in mind when interpreting statistics or training models.

In [ ]:
print(f"Mean:   {df['Age'].mean():.1f}")
print(f"Median: {df['Age'].median():.1f}")

df['Age'].plot(kind='hist', bins=40, figsize=(8, 3), title='Age distribution (before imputation)')
plt.show()

In [ ]:
df['Age'].fillna(df['Age'].median(), inplace=True)
print(f"NaN remaining in Age: {df['Age'].isna().sum()}")

### Strategy 3 — Drop rows

When very few rows are affected, dropping them is safe — you lose negligible data. Avoid this when missing values are numerous; use imputation or the keep strategy instead.

In [ ]:
df.dropna(inplace=True)   # only Embarked (2 rows) remains
print(f"Rows after dropping NaN: {len(df)}")
df.isna().sum()

---

## Part B — Placeholder Values

Not all missing data appears as `NaN`. Datasets often encode missing values with a placeholder that is technically valid — `?`, `-99`, `999`, `N/A` as a string. These won't show up in `.isna()` and must be found manually.

**For categorical columns:** inspect `.unique()` or `.value_counts()` and look for values that don't make sense.

**For numerical columns:** look for outliers or a single value appearing far more often than it should.

In [ ]:
# Check all categorical-looking columns for unexpected values
for col in ['Embarked', 'Sex', 'Pclass', 'Survived']:
    print(f"{col}: {df[col].unique()}")

`Pclass` contains `'?'` — a placeholder for unknown passenger class. Check how prevalent it is.

In [ ]:
df['Pclass'].value_counts(normalize=True).round(3)

About 5% of rows have an unknown class. Since `Pclass` is a key feature, dropping those rows would lose too much. Instead, impute by sampling from the observed class distribution.

In [ ]:
# Build the distribution from real values only
known = df[df['Pclass'] != '?']['Pclass'].value_counts(normalize=True)
classes, probs = known.index.tolist(), known.values.tolist()

df['Pclass'] = df['Pclass'].map(
    lambda x: np.random.choice(classes, p=probs) if x == '?' else x
)

print("After imputation:")
df['Pclass'].value_counts(normalize=True).round(3)

---

## Part C — Duplicates

Duplicate rows pass silently through `.isna()` checks but can inflate statistics and bias models. Load a messier version of the Titanic dataset that has intentional duplicates added.

In [ ]:
df2 = pd.read_csv('data/more_on_missing_data/titanic.csv')
df2.info()

In [ ]:
# Exact duplicate rows
exact_dupes = df2[df2.duplicated()]
print(f"Exact duplicate rows: {len(exact_dupes)}")
exact_dupes.head(3)

In [ ]:
# Key-based duplicates — rows that share a PassengerId (should be unique)
key_dupes = df2[df2.duplicated(subset='PassengerId')]
print(f"Rows with a duplicate PassengerId: {len(key_dupes)}")
key_dupes.tail(3)

Key-based duplicates are subtler — the rows are not identical but they share an ID that should be unique. This can indicate a data entry error, a bad join upstream, or a data export bug.

### Scanning all columns for anomalies

A quick pass through `.value_counts()` on every column surfaces placeholder values, unexpected dominant values, and category typos.

In [ ]:
for col in df2.columns:
    top = df2[col].value_counts(normalize=True).head(3)
    print(f"\n{col}:")
    print(top.round(3).to_string())

Notice that `PassengerId` value `839.0` appears in **29%** of rows — a clear sign of the intentional duplication. `Pclass` again has `?` accounting for ~10% of the data.

---

## Part D — Choosing a Strategy

There is no single right answer — the best approach depends on how much data is missing, the data type, and the downstream use.

| | Continuous | Categorical |
|---|---|---|
| **Delete rows** | OK when < ~5% affected | OK when < ~5% affected |
| **Delete column** | When > ~50–70% missing | When > ~50–70% missing |
| **Replace** | Median (robust) or mean (symmetric distributions) | Mode, if one category clearly dominates |
| **Keep as category** | Bin the column; missing becomes its own bin | Add `'Unknown'` / `'Missing'` as a valid category |

**Rules of thumb:**

- Prefer imputing over dropping — you keep more data.
- Imputing the median/mean reduces variance. If variance matters for your model, note the trade-off.
- When missingness itself is informative (e.g. a customer never clicked), keeping `NaN` as its own category preserves that signal.
- Drop columns only as a last resort; they may still contribute weak signal.

---

## Summary

In this notebook you learned how to:

- Detect `NaN` values with `.isna().sum()` and understand what drives each handling choice
- Drop columns (high % missing), impute with the median (continuous), and drop rows (very few affected)
- Find placeholder values like `?` using `.unique()` and `.value_counts()`
- Detect exact and key-based duplicate rows with `.duplicated()`
- Apply the continuous/categorical decision table to choose the right strategy

Next: [04 — Groupby and Aggregation](04_groupby_and_aggregation.ipynb)